# Track B — IFEval Constraint Extraction Experiment

**Goal**: Validate regex-based constraint extraction before wiring into `_score_structure_compliance`.

**Constraint types tested**:
- Exact item count: "exactly 5 items", "3 bullet points"
- Max word count: "under 200 words", "no more than 100 words"
- Min word count: "at least 150 words"
- Section count: "3 sections", "4 headings"
- Format type: bullet list, numbered list, JSON, table

In [ ]:
import re
from dataclasses import dataclass, field
from typing import Optional

# ── Constraint data structure ─────────────────────────────────────────────────

@dataclass
class Constraint:
    kind: str          # exact_count | max_words | min_words | section_count | bullet_list | numbered_list | json_format | table_format
    value: Optional[int] = None   # numeric value for count/word constraints
    raw_text: str = ""            # matched phrase

    def __repr__(self):
        v = f"={self.value}" if self.value else ""
        return f"Constraint({self.kind}{v}, from: '{self.raw_text}')"

print("Imports OK")

In [ ]:
# ── Constraint extractor ─────────────────────────────────────────────────────

_EXTRACTION_PATTERNS = [
    # ── Exact item count ──────────────────────────────────────────────────────
    (r"exactly\s+(\d+)\s+(?:bullet\s+points?|items?|steps?|recommendations?|findings?|examples?|points?)", "exact_count"),
    (r"(\d+)\s+bullet\s+points?", "exact_count"),
    (r"(\d+)\s+numbered\s+(?:steps?|items?|points?|recommendations?)", "exact_count"),
    (r"provide\s+(\d+)\s+(?:specific\s+)?(?:items?|steps?|recommendations?|examples?|reasons?|factors?)", "exact_count"),
    (r"list\s+(\d+)\s+(?:items?|steps?|recommendations?|examples?|reasons?|key\s+\w+)", "exact_count"),
    (r"(\d+)[\-–]point\s+(?:plan|summary|list|breakdown|analysis)", "exact_count"),

    # ── Max word count ────────────────────────────────────────────────────────
    (r"(?:under|within|below)\s+(\d+)\s+words?", "max_words"),
    (r"no\s+more\s+than\s+(\d+)\s+words?", "max_words"),
    (r"(?:max(?:imum)?|limit(?:ed)?\s+to)\s+(\d+)\s+words?", "max_words"),
    (r"(\d+)[- ]word\s+(?:limit|maximum|max|summary|response|answer)", "max_words"),
    (r"keep\s+(?:it\s+)?(?:to\s+)?(?:under\s+)?(\d+)\s+words?", "max_words"),

    # ── Min word count ────────────────────────────────────────────────────────
    (r"at\s+least\s+(\d+)\s+words?", "min_words"),
    (r"(?:min(?:imum)?)\s+(\d+)\s+words?", "min_words"),
    (r"(?:no\s+(?:less|fewer)\s+than|more\s+than)\s+(\d+)\s+words?", "min_words"),

    # ── Section count ─────────────────────────────────────────────────────────
    (r"(\d+)\s+(?:distinct\s+)?sections?", "section_count"),
    (r"(\d+)\s+(?:main\s+)?parts?", "section_count"),
    (r"(\d+)\s+headings?", "section_count"),
    (r"divide\s+(?:it\s+)?into\s+(\d+)", "section_count"),
    (r"organized?\s+(?:into|as)\s+(\d+)\s+(?:sections?|parts?|chapters?)", "section_count"),

    # ── Format type (no numeric value) ───────────────────────────────────────
    (r"(?:use\s+(?:a\s+)?)?bullet(?:ed)?\s+list", "bullet_list"),
    (r"(?:as\s+)?(?:a\s+)?(?:bulleted|unordered)\s+list", "bullet_list"),
    (r"(?:use\s+(?:a\s+)?)?numbered\s+list", "numbered_list"),
    (r"(?:as\s+)?(?:an?\s+)?ordered\s+list", "numbered_list"),
    (r"(?:in\s+)?JSON\s+(?:format|schema|output)?", "json_format"),
    (r"(?:as\s+(?:a\s+)?)?(?:markdown\s+)?table(?:\s+format)?", "table_format"),
]


def extract_constraints(text: str) -> list[Constraint]:
    """Extract all verifiable constraints from directive/context text."""
    lower = text.lower()
    found = []
    seen_kinds_with_values = set()

    for pattern, kind in _EXTRACTION_PATTERNS:
        for m in re.finditer(pattern, lower):
            value = int(m.group(1)) if m.lastindex and m.lastindex >= 1 else None
            key = (kind, value)
            if key not in seen_kinds_with_values:
                seen_kinds_with_values.add(key)
                found.append(Constraint(kind=kind, value=value, raw_text=m.group(0)))

    return found


print("Extractor defined. Testing...")
sample = "Provide exactly 5 bullet points, keep the response under 200 words, organized into 3 sections."
constraints = extract_constraints(sample)
for c in constraints:
    print(" ", c)

In [ ]:
# ── Verification logic ────────────────────────────────────────────────────────

def verify_constraint(c: Constraint, output: str) -> tuple[bool, str]:
    """Check whether an output satisfies a constraint. Returns (passed, evidence)."""
    wc = len(output.split())

    if c.kind == "exact_count":
        # Count list items (bullets or numbered)
        items = re.findall(r"(?:^|\n)\s*(?:[-*•]|\d+[.\)])\s+\S", output, re.MULTILINE)
        n_items = len(items)
        passed = (n_items == c.value)
        return passed, f"list items found: {n_items}, expected: {c.value}"

    elif c.kind == "max_words":
        passed = (wc <= c.value)
        return passed, f"word count: {wc}, limit: {c.value}"

    elif c.kind == "min_words":
        passed = (wc >= c.value)
        return passed, f"word count: {wc}, minimum: {c.value}"

    elif c.kind == "section_count":
        headings = re.findall(r"(?:^|\n)#{1,3}\s+\S|(?:^|\n)[A-Z][A-Za-z ]{3,30}:\s*\n", output, re.MULTILINE)
        n_headings = len(headings)
        passed = (n_headings >= c.value)
        return passed, f"headings found: {n_headings}, expected: {c.value}"

    elif c.kind == "bullet_list":
        has_bullets = bool(re.search(r"(?:^|\n)\s*[-*•]\s+\S", output, re.MULTILINE))
        return has_bullets, "bullet list present" if has_bullets else "no bullet list found"

    elif c.kind == "numbered_list":
        has_numbered = bool(re.search(r"(?:^|\n)\s*\d+[.\)]\s+\S", output, re.MULTILINE))
        return has_numbered, "numbered list present" if has_numbered else "no numbered list found"

    elif c.kind == "json_format":
        has_json = bool(re.search(r'\{[^{}]*"[a-z_A-Z]+"\s*:', output))
        return has_json, "JSON detected" if has_json else "no JSON found"

    elif c.kind == "table_format":
        has_table = bool(re.search(r"\|.+\|", output))
        return has_table, "table detected" if has_table else "no table found"

    return False, "unknown constraint type"


# Quick test
test_output = """Here are the 5 key findings:
- Performance declined 23% YoY
- Root cause: database query inefficiency
- Affected services: auth, payments, reports
- Timeline: started Q2 2024
- Immediate fix: add index on user_id column
"""
c = Constraint(kind="exact_count", value=5, raw_text="exactly 5 bullet points")
passed, evidence = verify_constraint(c, test_output)
print(f"exact_count=5: passed={passed} | {evidence}")

c2 = Constraint(kind="max_words", value=100, raw_text="under 100 words")
p2, e2 = verify_constraint(c2, test_output)
print(f"max_words=100: passed={p2} | {e2}")

In [ ]:
# ── Test dataset: 30 directive examples with expected extractions ─────────────

TEST_DIRECTIVES = [
    # exact_count
    ("Provide exactly 5 bullet points summarizing the issue.", [("exact_count", 5)]),
    ("List 3 root causes with supporting evidence.", [("exact_count", 3)]),
    ("Give me a 4-point plan for resolving this.", [("exact_count", 4)]),
    ("Identify exactly 7 risk factors.", [("exact_count", 7)]),
    ("Provide 6 numbered recommendations.", [("exact_count", 6)]),
    ("Summarize in 3 bullet points.", [("exact_count", 3)]),

    # max_words
    ("Respond in under 150 words.", [("max_words", 150)]),
    ("Keep the response to no more than 200 words.", [("max_words", 200)]),
    ("Write a 100-word summary.", [("max_words", 100)]),
    ("Maximum 300 words.", [("max_words", 300)]),
    ("Keep it within 50 words.", [("max_words", 50)]),

    # min_words
    ("Write at least 250 words.", [("min_words", 250)]),
    ("Provide a minimum 500 word analysis.", [("min_words", 500)]),
    ("No fewer than 100 words.", [("min_words", 100)]),

    # section_count
    ("Organize your response into 3 sections.", [("section_count", 3)]),
    ("Divide the analysis into 4 parts.", [("section_count", 4)]),
    ("Use 2 headings: Problem and Solution.", [("section_count", 2)]),
    ("Structured into 5 distinct sections.", [("section_count", 5)]),

    # bullet_list
    ("Use a bullet list to present your findings.", [("bullet_list", None)]),
    ("Present as a bulleted list.", [("bullet_list", None)]),

    # numbered_list
    ("Provide a numbered list of recommendations.", [("numbered_list", None)]),
    ("Use an ordered list.", [("numbered_list", None)]),

    # json_format
    ("Respond in JSON format.", [("json_format", None)]),
    ("Output as JSON schema.", [("json_format", None)]),

    # table_format
    ("Present as a markdown table.", [("table_format", None)]),
    ("Format your response as a table.", [("table_format", None)]),

    # multi-constraint
    ("Provide exactly 5 bullet points in under 200 words.", [("exact_count", 5), ("max_words", 200)]),
    ("List 3 findings organized into 2 sections.", [("exact_count", 3), ("section_count", 2)]),
    ("Numbered list, at least 100 words, max 300 words.", [("numbered_list", None), ("min_words", 100), ("max_words", 300)]),

    # negative control — no constraints
    ("Analyze the root cause of the performance degradation.", []),
]

print(f"Test dataset: {len(TEST_DIRECTIVES)} directives")

In [ ]:
# ── Extraction evaluation ─────────────────────────────────────────────────────

import pandas as pd

results = []
for directive, expected in TEST_DIRECTIVES:
    extracted = extract_constraints(directive)
    extracted_set = {(c.kind, c.value) for c in extracted}
    expected_set = set(expected)

    tp = len(extracted_set & expected_set)
    fp = len(extracted_set - expected_set)
    fn = len(expected_set - extracted_set)

    results.append({
        "directive": directive[:60] + ("..." if len(directive) > 60 else ""),
        "expected":  str(sorted(expected_set)),
        "extracted": str(sorted(extracted_set)),
        "TP": tp, "FP": fp, "FN": fn,
        "correct": (extracted_set == expected_set),
    })

df = pd.DataFrame(results)

# Overall stats
total_tp = df["TP"].sum()
total_fp = df["FP"].sum()
total_fn = df["FN"].sum()
precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
exact_match = df["correct"].mean()

print(f"Extraction results across {len(TEST_DIRECTIVES)} directives:")
print(f"  Precision:   {precision:.1%}")
print(f"  Recall:      {recall:.1%}")
print(f"  F1:          {f1:.1%}")
print(f"  Exact match: {exact_match:.1%}")
print()

# Show failures
failures = df[~df["correct"]]
if len(failures) == 0:
    print("All directives extracted correctly!")
else:
    print(f"Failures ({len(failures)}):")
    for _, row in failures.iterrows():
        print(f"  Directive: {row['directive']}")
        print(f"  Expected:  {row['expected']}")
        print(f"  Got:       {row['extracted']}")
        print()

In [ ]:
# ── Verification evaluation: test constraint checking on synthetic outputs ────

VERIFICATION_TESTS = [
    # (constraint, output_text, expected_pass)

    # exact_count — correct
    (Constraint("exact_count", 5, "exactly 5"),
     "- item one\n- item two\n- item three\n- item four\n- item five",
     True),

    # exact_count — wrong count
    (Constraint("exact_count", 5, "exactly 5"),
     "- item one\n- item two\n- item three",
     False),

    # max_words — passes
    (Constraint("max_words", 50, "under 50 words"),
     "The system failed due to a memory leak in the authentication module.",
     True),

    # max_words — fails
    (Constraint("max_words", 10, "under 10 words"),
     "The system failed due to a memory leak in the authentication module that was introduced in the latest deployment.",
     False),

    # min_words — passes
    (Constraint("min_words", 5, "at least 5 words"),
     "The root cause is a memory leak.",
     True),

    # min_words — fails
    (Constraint("min_words", 100, "at least 100 words"),
     "Short answer.",
     False),

    # section_count — passes
    (Constraint("section_count", 2, "2 sections"),
     "## Problem\nDetails here.\n\n## Solution\nFix here.",
     True),

    # section_count — fails
    (Constraint("section_count", 3, "3 sections"),
     "## Problem\nDetails here.\n\n## Solution\nFix here.",
     False),

    # bullet_list — passes
    (Constraint("bullet_list", None, "bullet list"),
     "- First point\n- Second point\n- Third point",
     True),

    # bullet_list — fails
    (Constraint("bullet_list", None, "bullet list"),
     "This is a plain paragraph with no list formatting.",
     False),

    # numbered_list — passes
    (Constraint("numbered_list", None, "numbered list"),
     "1. Step one\n2. Step two\n3. Step three",
     True),

    # json_format — passes
    (Constraint("json_format", None, "JSON format"),
     '{"status": "error", "code": 500, "message": "Internal server error"}',
     True),

    # json_format — fails
    (Constraint("json_format", None, "JSON format"),
     "The status is error with code 500.",
     False),

    # table_format — passes
    (Constraint("table_format", None, "table"),
     "| Name | Score |\n|------|-------|\n| Alice | 95 |",
     True),
]

ver_results = []
for c, output, expected_pass in VERIFICATION_TESTS:
    passed, evidence = verify_constraint(c, output)
    correct = (passed == expected_pass)
    ver_results.append({
        "constraint": repr(c),
        "expected_pass": expected_pass,
        "actual_pass": passed,
        "correct": correct,
        "evidence": evidence,
    })

vdf = pd.DataFrame(ver_results)
accuracy = vdf["correct"].mean()
print(f"Verification accuracy: {accuracy:.1%} ({vdf['correct'].sum()}/{len(vdf)} tests passed)")
print()

failures = vdf[~vdf["correct"]]
if len(failures) == 0:
    print("All verification tests passed!")
else:
    print(f"Verification failures ({len(failures)}):")
    for _, row in failures.iterrows():
        print(f"  Constraint: {row['constraint']}")
        print(f"  Expected pass={row['expected_pass']}, got pass={row['actual_pass']}")
        print(f"  Evidence: {row['evidence']}")
        print()

In [ ]:
# ── Scoring simulation: what would the new _score_structure_compliance produce? ─

def simulate_structure_score(directive: str, output: str) -> dict:
    """Simulate the new IFEval-aware scoring vs the current keyword-only scoring."""
    lower_ctx = directive.lower()
    lower_out = output.lower()

    # ── Current scorer (keyword-only) ────────────────────────────────────────
    current_score = 0.4
    notes_current = []

    wants_json = "json" in lower_ctx
    has_json = bool(re.search(r'\{[^{}]*"[a-z_]+"\s*:', output))
    if wants_json:
        current_score += 0.4 if has_json else -0.2
        notes_current.append("JSON " + ("found" if has_json else "missing"))

    wants_list = any(p in lower_ctx for p in ["bullet", "numbered list", "list of"])
    has_list = bool(re.findall(r"\n\s*[-*]\s|\n\s*\d+[\.\)]\s", output))
    if wants_list:
        current_score += 0.3 if has_list else -0.1
        notes_current.append("List " + ("found" if has_list else "missing"))

    wants_headers = any(p in lower_ctx for p in ["sections", "headings", "## "])
    has_headers = "##" in output
    if wants_headers:
        current_score += 0.3 if has_headers else -0.1
        notes_current.append("Headers " + ("found" if has_headers else "missing"))

    if not wants_json and not wants_list and not wants_headers:
        if has_headers and has_list:
            current_score += 0.35
        elif has_headers or has_list:
            current_score += 0.20

    current_score = max(0.0, min(1.0, current_score))

    # ── New IFEval scorer ──────────────────────────────────────────────────────
    new_score = current_score  # starts from current
    constraints = extract_constraints(directive)
    notes_new = list(notes_current)
    satisfied = 0
    violated = 0

    for c in constraints:
        passed, evidence = verify_constraint(c, output)
        if passed:
            satisfied += 1
            notes_new.append(f"✓ {c.kind}={c.value}: {evidence}")
        else:
            violated += 1
            notes_new.append(f"✗ {c.kind}={c.value}: {evidence}")

    bonus = min(0.45, satisfied * 0.15)
    penalty = violated * 0.20
    new_score = max(0.0, min(1.0, new_score + bonus - penalty))

    return {
        "constraints_found": len(constraints),
        "constraints_satisfied": satisfied,
        "constraints_violated": violated,
        "current_score": round(current_score, 3),
        "new_score": round(new_score, 3),
        "delta": round(new_score - current_score, 3),
        "notes": notes_new,
    }


# Test scenarios
scenarios = [
    {
        "label": "Word count satisfied",
        "directive": "Analyze the root cause. Keep response under 200 words.",
        "output": "The root cause is a memory leak. " * 5,  # ~30 words
    },
    {
        "label": "Word count violated",
        "directive": "Provide a brief summary under 20 words.",
        "output": "The root cause is a memory leak in the authentication module introduced in the latest deployment which caused performance degradation across the platform affecting all users.",
    },
    {
        "label": "Exact count satisfied",
        "directive": "List exactly 3 recommendations.",
        "output": "- Restart the service\n- Add monitoring\n- Increase memory limit",
    },
    {
        "label": "Exact count violated",
        "directive": "Provide exactly 5 bullet points.",
        "output": "- Point one\n- Point two\n- Point three",
    },
    {
        "label": "Multi-constraint all satisfied",
        "directive": "Provide exactly 3 bullet points in under 100 words.",
        "output": "- First finding: memory leak\n- Second finding: slow queries\n- Third finding: missing indexes",
    },
    {
        "label": "No constraints in directive",
        "directive": "Analyze the performance degradation.",
        "output": "The system is experiencing slowdowns due to increased load.",
    },
]

print(f"{'Scenario':<35} {'Current':>8} {'New':>8} {'Delta':>8} {'Constraints'}")
print("-" * 75)
for s in scenarios:
    r = simulate_structure_score(s["directive"], s["output"])
    print(f"{s['label']:<35} {r['current_score']:>8.3f} {r['new_score']:>8.3f} {r['delta']:>+8.3f}  {r['constraints_satisfied']}/{r['constraints_found']} satisfied")

## Conclusions

**Extraction precision/recall**: Run cell 5 to see results. Target: F1 > 0.85.

**Verification accuracy**: Run cell 6. Target: > 90%.

**Score delta**: The new scorer should:
- Reward satisfied constraints with +0.15 each (capped at +0.45)
- Penalize violated constraints with −0.20 each
- Leave score unchanged when no verifiable constraints found (safe for all existing use cases)

**Deployment decision**:
- If extraction F1 > 0.85 AND verification accuracy > 90% → ready to deploy in `_score_structure_compliance`
- If below targets → identify failing patterns, refine regex, re-test

**Next**: If validated here, implement in `src/mycontext/intelligence/output_evaluator.py`